In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;

In [0]:
from pyspark.sql.functions import col, trim

# ================================
# 1. READ BRONZE TABLE
# ================================

df = spark.table("electronics_retailer_clg.bronze.exchange_rates")


# ================================
# 2. CLEAN COLUMN NAMES
# ================================

df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])


# ================================
# 3. TRIM SPACES (DATA QUALITY CHECK)
# ================================

for c in df.columns:
    df = df.withColumn(c, trim(col(c)))


# ================================
# 4. REMOVE UNUSED COLUMN (DATE) 
# ================================

df = df.drop("date")


# ================================
# 5. FIX DATA TYPE (IMPORTANT)
# ================================

df = df.withColumn("exchange", col("exchange").cast("double"))


# ================================
# 6. FINAL CHECK
# ================================

display(df)
df.printSchema()


# ================================
# 7. WRITE TO SILVER
# ================================

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("electronics_retailer_clg.silver.exchange_rates")

print("Exchange rates cleaned successfully")